In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from dataclasses import dataclass

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage.feature import blob_log

from pkg.selection_window import CircleSelectionWindow

In [ ]:
@dataclass
class BlobParams:
    min_sigma: int = 1
    max_sigma: int = 50
    num_sigma: int = 10
    threshold: float = 0.2
    overlap: float = 0.5

In [ ]:
def run_detection(
    img: np.ndarray,
    center: tuple,
    radius: int,
    small_blob_params: BlobParams,
    large_blob_params: BlobParams,
    num_large_blobs: int = 1,
    rim_radius_ratio: float = 0.9,
    rim_size: int = 5,
    remove_small_blobs_inside_large: bool = True,
):
    small_blobs = blob_log(img, **small_blob_params.__dict__)
    large_blobs = blob_log(img, **large_blob_params.__dict__)

    # Convert sigma to radius
    if len(small_blobs) > 0:
        small_blobs[:, 2] *= np.sqrt(2)

    if len(large_blobs) > 0:
        large_blobs[:, 2] *= np.sqrt(2)

    # Keep n largest blobs
    large_blobs = sorted(large_blobs, key=lambda b: b[2], reverse=True)[
        :num_large_blobs
    ]

    # Filter small blobs
    filtered_small = []

    for y, x, r in small_blobs:
        if r < 2:
            continue

        if remove_small_blobs_inside_large:
            # reject small blobs that lie inside / strongly overlap large blobs
            is_inside_large_blob = False
            for large_blob in large_blobs:
                yL, xL, rL = large_blob
                d = np.hypot(x - xL, y - yL)
                if d < rL:
                    is_inside_large_blob = True
                    break

            if is_inside_large_blob:
                continue

        if (x - center[0]) ** 2 + (
            y - center[1]
        ) ** 2 > radius**2 * rim_radius_ratio and r < rim_size:
            continue

        filtered_small.append((y, x, r))

    return filtered_small, large_blobs


def plot_small_blobs(img: np.ndarray, small_blobs, large_blobs):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax = plt.gca()

    for y, x, r in small_blobs:
        c = plt.Circle((x, y), r, color="red", linewidth=1.5, fill=False)
        ax.add_patch(c)

    for large_blob in large_blobs:
        y, x, r = large_blob
        c = plt.Circle((x, y), r, color="yellow", linewidth=2.5, fill=False)
        ax.add_patch(c)

    ax.axis("off")
    fig.tight_layout()
    return fig, ax

## Test Image

In [ ]:
img = cv2.imread("data/colony_log/test.jpg")

window = CircleSelectionWindow("Circle Selection Window", img)
window.displayWindow()
circle = window.get_circle()
print(circle)

In [ ]:
circle = (479.3195033235295, 482.46711387399506, np.float64(424.99930765494565))
center = (int(round(circle[0])), int(round(circle[1])))
radius = int(round(circle[2]))

# Set pixels outside the circle to black
mask = cv2.circle(
    np.zeros(img.shape[:2], dtype=np.uint8), center, radius, 255, thickness=-1
)
masked_img = cv2.bitwise_and(img, img, mask=mask)

# Convert to grayscale and apply CLAHE
gray = cv2.cvtColor(masked_img, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))
img_eq = clahe.apply(gray)
plt.imshow(img_eq, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
small_blob_params = BlobParams(min_sigma=2, max_sigma=8, num_sigma=40, threshold=0.07)
large_blob_params = BlobParams(min_sigma=10, max_sigma=40, num_sigma=40, threshold=0.3)
small_blobs, large_blobs = run_detection(
    img_eq, center, radius, small_blob_params, large_blob_params
)
fig, ax = plot_small_blobs(img, small_blobs, large_blobs)
fig.savefig("presentation/detected_colonies.png", bbox_inches="tight", pad_inches=0)
plt.show()